In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
data = pd.read_excel('dataset_gen.xlsx') 
dataset_all_gen = pd.read_excel('dataset_all_gen.xlsx')
dataset_all_original = pd.read_excel('dataset_all_original.xlsx')
dataset_jaccard_gen = pd.read_excel('dataset_jaccard_gen.xlsx')
dataset_jaccard_original = pd.read_excel('dataset_jaccard_original.xlsx')
dataset_myroom_gen = pd.read_excel('dataset_myroom_gen.xlsx')
dataset_myroom_original = pd.read_excel('dataset_myroom_original.xlsx')
dataset_university_gen = pd.read_excel('dataset_university_gen.xlsx')
dataset_university_original = pd.read_excel('dataset_university_original.xlsx')

In [4]:

data = data.sample(frac = 1) 
dataset_all_gen = dataset_all_gen.sample(frac = 1) 
dataset_all_original = dataset_all_original.sample(frac = 1) 
dataset_jaccard_gen = dataset_jaccard_gen.sample(frac = 1) 
dataset_jaccard_original = dataset_jaccard_original.sample(frac = 1) 
dataset_myroom_gen = dataset_myroom_gen.sample(frac = 1) 
dataset_myroom_original = dataset_myroom_original.sample(frac = 1) 
dataset_university_gen = dataset_university_gen.sample(frac = 1) 
dataset_university_original = dataset_university_original.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)
dataset_all_gen = dataset_all_gen.reset_index(drop=True)
dataset_all_original = dataset_all_original.reset_index(drop=True)
dataset_jaccard_gen = dataset_jaccard_gen.reset_index(drop=True)
dataset_jaccard_original = dataset_jaccard_original.reset_index(drop=True)
dataset_myroom_gen = dataset_myroom_gen.reset_index(drop=True)
dataset_myroom_original = dataset_myroom_original.reset_index(drop=True)
dataset_university_gen = dataset_university_gen.reset_index(drop=True)
dataset_university_original = dataset_university_original.reset_index(drop=True)

In [6]:
data.isnull().sum()

type             0
Name             0
Vendor           0
DNS           2304
mDNS          2704
NUPNP        11000
open_port     1731
SSDP_Type     2304
SSDP_t       11000
SSDP_v       11000
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
#fill Na
dataset_all_gen = dataset_all_gen.fillna("NaN")
dataset_all_original = dataset_all_original.fillna("NaN")
dataset_jaccard_gen = dataset_jaccard_gen.fillna("NaN")
dataset_jaccard_original = dataset_jaccard_original.fillna("NaN")
dataset_myroom_gen = dataset_myroom_gen.fillna("NaN")
dataset_myroom_original = dataset_myroom_original.fillna("NaN")
dataset_university_gen = dataset_university_gen.fillna("NaN")
dataset_university_original = dataset_university_original.fillna("NaN")

In [9]:
data.isnull().sum()

type         0
Name         0
Vendor       0
DNS          0
mDNS         0
NUPNP        0
open_port    0
SSDP_Type    0
SSDP_t       0
SSDP_v       0
dtype: int64

In [10]:
data.dtypes

type         object
Name         object
Vendor       object
DNS          object
mDNS         object
NUPNP        object
open_port    object
SSDP_Type    object
SSDP_t       object
SSDP_v       object
dtype: object

In [11]:
data = data.astype('category')

In [12]:
dataset_all_gen = dataset_all_gen.astype('category')
dataset_all_original = dataset_all_original.astype('category')
dataset_jaccard_gen = dataset_jaccard_gen.astype('category')
dataset_jaccard_original = dataset_jaccard_original.astype('category')
dataset_myroom_gen = dataset_myroom_gen.astype('category')
dataset_myroom_original = dataset_myroom_original.astype('category')
dataset_university_gen = dataset_university_gen.astype('category')
dataset_university_original = dataset_university_original.astype('category')

In [13]:
data.dtypes

type         category
Name         category
Vendor       category
DNS          category
mDNS         category
NUPNP        category
open_port    category
SSDP_Type    category
SSDP_t       category
SSDP_v       category
dtype: object

In [14]:
data

,type,Name,Vendor,DNS,mDNS,NUPNP,open_port,SSDP_Type,SSDP_t,SSDP_v
0,Appliances,Netatmo Welcome,Netatmo,NaN,NaN,NaN,tcp/1883,NaN,NaN,NaN
1,Appliances,Triby Speaker Smart Speaker,Invoxia,marriott.com,_sonair._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
2,Health-Monitor,Blipcare Blood Pressure meter,Blipcare,theguardian.com,_tempsoft._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
3,Non-IOT,Macbook3,"Apple, Inc.",scribd.com,NaN,NaN,tcp/80,M-SEARCH * HTTP/1.1,NaN,NaN
4,Non-IOT,Samsung s7,SAMSUNG,indiegogo.com,_lotstring._tcp.local.,NaN,tcp/1900,M-SEARCH * HTTP/1.1,NaN,NaN
5,Cameras,NestCam,Nest Labs Inc.,nifty.com,_redhold._tcp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NaN,NaN
6,Hubs,Apple HomePod,"Apple, Inc.",wikia.com,_toughjoyfax._tcp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,NaN,NaN
7,Non-IOT,Dell Win7 laptop,Dell,alibaba.com,NaN,NaN,tcp/443,M-SEARCH * HTTP/1.1,NaN,NaN
8,Energy Management,Belkin wemo motion sensor,Belkin International Inc.,deviantart.com,_temp._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
9,Non-IOT,Xiaomi smartphone,Xiaomi,blogtalkradio.com,NaN,NaN,tcp/80,M-SEARCH * HTTP/1.1,NaN,NaN


In [15]:
y = data.drop_duplicates(keep='first')
y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 
y

,type,Vendor,DNS,mDNS,NUPNP,open_port,SSDP_Type,SSDP_t,SSDP_v
0,Appliances,Netatmo,NaN,NaN,NaN,tcp/1883,NaN,NaN,NaN
1,Appliances,Invoxia,marriott.com,_sonair._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
2,Health-Monitor,Blipcare,theguardian.com,_tempsoft._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
3,Non-IOT,"Apple, Inc.",scribd.com,NaN,NaN,tcp/80,M-SEARCH * HTTP/1.1,NaN,NaN
4,Non-IOT,SAMSUNG,indiegogo.com,_lotstring._tcp.local.,NaN,tcp/1900,M-SEARCH * HTTP/1.1,NaN,NaN
5,Cameras,Nest Labs Inc.,nifty.com,_redhold._tcp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NaN,NaN
6,Hubs,"Apple, Inc.",wikia.com,_toughjoyfax._tcp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,NaN,NaN
7,Non-IOT,Dell,alibaba.com,NaN,NaN,tcp/443,M-SEARCH * HTTP/1.1,NaN,NaN
8,Energy Management,Belkin International Inc.,deviantart.com,_temp._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
9,Non-IOT,Xiaomi,blogtalkradio.com,NaN,NaN,tcp/80,M-SEARCH * HTTP/1.1,NaN,NaN


In [16]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original.drop_duplicates(keep='first')

In [17]:
#drop name
dataset_all_gen_y = dataset_all_gen_y.drop(['Name'], axis=1)
dataset_all_original_y = dataset_all_original_y.drop(['Name'], axis=1)
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop(['Name'], axis=1)
dataset_jaccard_original_y = dataset_jaccard_original_y.drop(['Name'], axis=1)
dataset_myroom_gen_y = dataset_myroom_gen_y.drop(['Name'], axis=1)
dataset_myroom_original_y = dataset_myroom_original_y.drop(['Name'], axis=1)
dataset_university_gen_y = dataset_university_gen_y.drop(['Name'], axis=1)
dataset_university_original_y = dataset_university_original_y.drop(['Name'], axis=1)

In [18]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen_y.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original_y.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original_y.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen_y.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original_y.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen_y.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original_y.drop_duplicates(keep='first')

In [19]:
y = y.apply(LabelEncoder().fit_transform)

In [20]:
dataset_all_gen_y = dataset_all_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_all_original_y = dataset_all_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_original_y = dataset_jaccard_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_gen_y = dataset_myroom_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_original_y = dataset_myroom_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_gen_y = dataset_university_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_original_y = dataset_university_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

In [21]:
list_data = []
list_data.append(y)
list_data.append(dataset_all_gen_y)
list_data.append(dataset_all_original_y)
list_data.append(dataset_jaccard_gen_y)
list_data.append(dataset_jaccard_original_y)
list_data.append(dataset_myroom_gen_y)
list_data.append(dataset_myroom_original_y)
list_data.append(dataset_university_gen_y)
list_data.append(dataset_university_original_y)
len(list_data)

9

In [22]:
accuracy = []
accuracy_10 = []
accuracy_temp = []

In [23]:
for data in list_data:
    for i in range(10):
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
        kf.get_n_splits(data)
        list_test = []
        for train_index, test_index in kf.split(data):
        #     print(index)
        #     print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
        #     print(len(train))
            list_test.append(test_index)
        #     print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index()
            clf = tree.DecisionTreeClassifier()
            clf = clf.fit(train,label_data )
            predict = clf.predict(test)
        #     len(predict)
            result_label_data = pd.DataFrame(columns = ['type'])
            xresult_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
        #     print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_10.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_10))

0   Accuracy:  1.0
0   Accuracy:  1.0
0   Accuracy:  1.0
0   Accuracy:  1.0
0   Accuracy:  1.0
0   Accuracy:  1.0
0   Accuracy:  1.0
0   Accuracy:  1.0
0   Accuracy:  1.0
0   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
1   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
2   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy

3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
3   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
4   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
5   Accuracy:  1.0
6   Accuracy:  1.0
6   Accuracy:  1.0
6   Accuracy:  1.0
6   Accuracy:  1.0
6   Accuracy:  1.0
6   Accuracy:  1.0
6   Accuracy:  1.0
6   Accuracy:  1.0
6   Accuracy:  1.0
6   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy

6   Accuracy:  1.0
6   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
7   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
8   Accuracy:  1.0
9   Accuracy:  1.0
9   Accuracy:  1.0
9   Accuracy:  1.0
9   Accuracy:  1.0
9   Accuracy:  1.0
9   Accuracy:  1.0
9   Accuracy:  1.0
9   Accuracy:  1.0
9   Accuracy:  1.0
9   Accuracy:  1.0


In [24]:
for t in accuracy:
    print("Accuracy for each dataset: ",t)

Accuracy for each dataset:  1.0
Accuracy for each dataset:  1.0
Accuracy for each dataset:  1.0
Accuracy for each dataset:  1.0
Accuracy for each dataset:  1.0
Accuracy for each dataset:  1.0
Accuracy for each dataset:  1.0
Accuracy for each dataset:  1.0
Accuracy for each dataset:  1.0
